# LAFM block-by-block workflow (no dashboard)

Run one block at a time. The workflow is: load → optional preprocessing → select one representative particle as the ROI Reference → cross-correlate that template against every complete frame → average detected particle crops → re-detect with the average reference → align/recalculate → localize/render → save.

In [ ]:
%matplotlib widget
from pathlib import Path
import importlib
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.widgets import RectangleSelector
import pnanolocz.lafm_workflow as workflow_module
importlib.reload(workflow_module)
from pnanolocz.lafm_workflow import LAFMWorkflow

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
IMAGE_DIR = PROJECT_ROOT / "Software_testing_images" / "LAFM testing"
TIFF_PATH = sorted(IMAGE_DIR.glob("*.tiff"))[0]  # change this file if needed
OUTPUT_DIR = PROJECT_ROOT / "Software_testing_images" / "test_output" / "lafm_block_by_block"
FRAME = 0
print("TIFF:", TIFF_PATH)

## 1. Load and optional preprocessing

Testing images default to no leveling and no filtering. Change either switch to `True` only when required.

In [ ]:
USE_LEVELING = False
USE_FILTERING = False

workflow = LAFMWorkflow.from_tiff(TIFF_PATH)
movie = workflow.preprocess(
    use_leveling=USE_LEVELING,
    level_routine="plane-line",
    use_filtering=USE_FILTERING,
    filter_name="Gaussian",
    filter_strength=1.0,
)
print("Movie shape (frames, rows, columns):", movie.shape)

## 2. Select ROI

Press the left mouse button on the image, tightly surround one representative particle, drag, and release. This crop becomes the ROI Reference; the movie itself is not cropped.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(movie[FRAME], cmap="afmhot", origin="upper", interpolation="nearest")
ax.set_title(f"Drag ROI — {TIFF_PATH.name}, frame {FRAME}")

def on_roi_selected(click, release):
    rows, cols = movie[FRAME].shape
    x0 = max(0, int(np.floor(min(click.xdata, release.xdata))))
    x1 = min(cols, int(np.ceil(max(click.xdata, release.xdata))))
    y0 = max(0, int(np.floor(min(click.ydata, release.ydata))))
    y1 = min(rows, int(np.ceil(max(click.ydata, release.ydata))))
    roi_reference = workflow.set_reference_roi((x0, y0, x1, y1), frame_index=FRAME)
    print("ROI Reference:", workflow.roi_bounds, roi_reference.shape)
    print("Detection movie remains complete:", workflow.roi_movie.shape)

roi_selector = RectangleSelector(
    ax, on_roi_selected, button=[1], interactive=True,
    minspanx=1, minspany=1, spancoords="pixels",
)
plt.show()

## 3. Detect particles in every frame

This uses the MATLAB ROI Method: template cross-correlation of the selected ROI Reference against every complete movie frame. It does not use Peak Method. Every row records `(x, y, z, correlation, frame, ...)`; frame numbers are MATLAB-style 1-based.

In [ ]:
SELECT_METHOD = "ROI"       # "ROI" (default) or "Peaks"
FAST_FIND = True
FILTER_XCOR_IMAGE = 1.0
FILTER_IMAGE = 1.0
EXCLUDE_EDGES = True
CORRELATION_MAX = 1.0
CORRELATION_MIN = 0.5
MAX_STEP = 10.0
MAX_MISSING = 3

initial_particles = workflow.detect_initial(
    method=SELECT_METHOD,
    image_filter_sigma=FILTER_IMAGE,
    correlation_filter_sigma=FILTER_XCOR_IMAGE,
    fast_find=FAST_FIND,
    exclude_edges=EXCLUDE_EDGES,
    correlation_min=CORRELATION_MIN,
    correlation_max=CORRELATION_MAX,
)
workflow.settings.update(tracking_max_step=MAX_STEP, tracking_max_missing=MAX_MISSING)
frames, counts = np.unique(initial_particles[:, 4].astype(int), return_counts=True)
print(f"Detected {len(initial_particles)} particles across {len(frames)}/{workflow.roi_movie.shape[0]} frames")
print("Particles per detected frame: min", counts.min(), "max", counts.max())
detected_particle = workflow.detected_particle_preview(crop_radius=5)
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(detected_particle, cmap="afmhot", interpolation="nearest")
ax.set_title("One detected particle crop — not averaged")
plt.show()

## 4. Average the detected particle images

Each `(x, y, frame)` row is used to crop that particle from its own frame. All valid crops are averaged into one reference image.

In [ ]:
average_reference = workflow.calculate_average_reference(crop_radius=5)
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(average_reference, cmap="afmhot", interpolation="nearest")
ax.set_title(f"Average reference from {len(workflow.initial_locs)} particle crops")
plt.show()

## 5. Re-detect particles using the average reference

In [ ]:
reference_particles = workflow.detect_with_reference(
    correlation_filter_sigma=0.5,
    threshold=0.3,
)
frames = np.unique(reference_particles[:, 4].astype(int))
print(f"Reference detection: {len(reference_particles)} particles across {len(frames)}/{workflow.roi_movie.shape[0]} frames")

## 6. Translation alignment and correlation recalculation

In [ ]:
aligned_reference = workflow.align_translation(
    iterations=2,
    method="Cross corr",
    max_drift=3.0,
    auto_update_reference=True,
)
recalculated_particles = workflow.recalculate_correlation(threshold=0.3)
print("Particles after recalculating correlation:", len(recalculated_particles))

## 7. Find all peaks / sub-pixel localization

Run and inspect the localization count before rendering.

In [ ]:
LOW_PASS_GAUSSIAN = 0.0
HIGH_PASS_OFF = 0.0
MIN_SEPARATION = 1
HEIGHT = 0.0
PROMINENCE = 0.0

localized = workflow.find_all_peaks(
    localization_method="cvcubic",
    pixperfeat=1.0,
    low_pass_sigma=LOW_PASS_GAUSSIAN,
    high_pass_sigma=HIGH_PASS_OFF,
    min_separation=MIN_SEPARATION,
    height_threshold=HEIGHT,
    prominence_threshold=PROMINENCE,
)
print("Localized particles:", len(localized))

## 8. Render LAFM

This block only renders the localization table created above.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from matplotlib.cm import ScalarMappable
from matplotlib.colors import ListedColormap, Normalize
from pnanolocz.lafm_workflow import COLORMAP_NAMES, resolve_lafm_colormap

finite_z = workflow.included_localizations[:, 2]
finite_z = finite_z[np.isfinite(finite_z)]
z_low, z_high = float(finite_z.min()), float(finite_z.max())
z_step = max((z_high - z_low) / 200.0, np.finfo(float).eps)
colormap_control = widgets.Dropdown(
    options=COLORMAP_NAMES, value="LAFM color", description="Colormap:"
)
lafm_z_control = widgets.FloatRangeSlider(
    value=(z_low, z_high), min=z_low, max=z_high, step=z_step,
    description="LAFM z range:", continuous_update=False,
    readout_format=".3g", layout=widgets.Layout(width="520px"),
)
probability_z_control = widgets.FloatRangeSlider(
    value=(z_low, z_high), min=z_low, max=z_high, step=z_step,
    description="Probability z range:", continuous_update=False,
    readout_format=".3g", layout=widgets.Layout(width="520px"),
)
display(widgets.VBox([colormap_control, lafm_z_control, probability_z_control]))

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
lafm_z_colorbar = None
probability_density_colorbar = None

def update_renders(_change=None):
    global lafm_rgb, lafm_display, probability, z_limits
    global lafm_z_colorbar, probability_density_colorbar
    cmap_array = resolve_lafm_colormap(colormap_control.value)
    display_cmap = ListedColormap(cmap_array, name=colormap_control.value)
    lafm_rgb, probability, z_limits = workflow.render_lafm(
        colormap_name=colormap_control.value,
        img_gus=1.0, expand=5.0, delete_outliers=4.0,
        colorlimits=(0.0, 1.0), colorlimit_mode="Max Min",
        lafm_z_range=tuple(lafm_z_control.value),
        probability_z_range=tuple(probability_z_control.value),
    )
    lafm_display = np.clip(
        lafm_rgb / max(float(np.nanmax(lafm_rgb)), 1.0), 0.0, 1.0
    )
    axes[0].clear(); axes[1].clear()
    axes[0].imshow(lafm_display, origin="upper")
    probability_artist = axes[1].imshow(
        probability, cmap=display_cmap, origin="upper"
    )
    axes[0].set_title(
        f"LAFM — {colormap_control.value}" if len(workflow.rendered_lafm_locs)
        else "LAFM — selected z range is empty"
    )
    axes[1].set_title(
        "Probability / localization density"
        if len(workflow.rendered_probability_locs)
        else "Probability — selected z range is empty"
    )
    if lafm_z_colorbar is not None: lafm_z_colorbar.remove()
    if probability_density_colorbar is not None: probability_density_colorbar.remove()
    z_map = ScalarMappable(
        norm=Normalize(float(z_limits[0]), float(z_limits[1])), cmap=display_cmap
    )
    lafm_z_colorbar = fig.colorbar(z_map, ax=axes[0], label="Height (z)")
    probability_density_colorbar = fig.colorbar(
        probability_artist, ax=axes[1], label="Localization density"
    )
    fig.canvas.draw_idle()

for control in (colormap_control, lafm_z_control, probability_z_control):
    control.observe(update_renders, names="value")
update_renders()
plt.show()

## 9. Save

In [ ]:
saved_paths = workflow.save_results(OUTPUT_DIR)
for name, path in saved_paths.items():
    print(f"{name}: {path}")